# 04. 예외 케이스 판정과 통합 정책

00~03에서 발견된 강도 인코딩, affine, z-spacing 예외를 한곳에 모읍니다. 이 노트북은 자동으로 원본을 고치지 않습니다. 검토자가 조치와 근거를 승인해야만 뒤 전처리 노트북이 해당 케이스를 처리합니다.

원본 NIfTI/DICOM 확인 없이 `approved=True`로 바꾸지 마세요.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'project_config.json').is_file() and (candidate / 'ctqa_utils.py').is_file():
            return candidate
    raise FileNotFoundError('project_config.json이 있는 프로젝트 폴더를 여세요.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from ctqa_utils import as_bool, write_json

CONFIG = json.loads((PROJECT_ROOT / 'project_config.json').read_text(encoding='utf-8'))
DATA_ROOT = Path(CONFIG['data_root'])
RESULTS = PROJECT_ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
print('프로젝트:', PROJECT_ROOT)
print('결과 저장:', RESULTS)

프로젝트: c:\workspace\hodgepodge\hy\데이터품질검증 및 통합
결과 저장: c:\workspace\hodgepodge\hy\데이터품질검증 및 통합\results


In [2]:
required = {
    'pairs': RESULTS / '00_pair_manifest.csv',
    'geometry': RESULTS / '01_geometry_checks.csv',
    'values': RESULTS / '02_value_label_checks.csv',
}
missing = [str(path) for path in required.values() if not path.is_file()]
if missing:
    raise RuntimeError('00~02 노트북을 먼저 실행하세요. 누락:\n' + '\n'.join(missing))
pairs = pd.read_csv(required['pairs'])
geometry = pd.read_csv(required['geometry'])
values = pd.read_csv(required['values'])
print('검토 대상 전체 케이스:', len(pairs))

검토 대상 전체 케이스: 267


## 1. 자동 탐지 결과를 검토 레지스트리로 변환

허용 조치는 이슈별로 제한합니다. `EXCLUDE`도 사유와 검토자 기록이 있어야 승인된 해결로 간주합니다.

In [3]:
findings = []
for row in values.loc[values.ct_encoding_candidate.astype(bool)].itertuples():
    evidence = {'min': row.ct_min, 'p50': row.ct_p50, 'p99': row.ct_p99, 'max': row.ct_max}
    findings.append({
        'source': row.source, 'case_id': row.case_id, 'issue': 'INTENSITY_SIGNED12',
        'evidence': json.dumps(evidence, ensure_ascii=False),
        'allowed_actions': 'APPLY_SIGNED12|KEEP_CONFIRMED|EXCLUDE',
    })

for row in geometry.loc[~geometry.affine_match.astype(bool)].itertuples():
    evidence = {'affine_max_abs_diff': row.affine_max_abs_diff, 'image_affine_source': row.image_affine_source, 'mask_affine_source': row.mask_affine_source}
    findings.append({
        'source': row.source, 'case_id': row.case_id, 'issue': 'AFFINE_MISMATCH',
        'evidence': json.dumps(evidence, ensure_ascii=False),
        'allowed_actions': 'COPY_CT_AFFINE_TO_MASK|RESAMPLE_MASK_TO_CT|EXCLUDE',
    })

z_outlier = pd.Series(False, index=geometry.index)
source_medians = geometry.groupby('source').z_spacing_mm.median().to_dict()
for source, idx in geometry.groupby('source').groups.items():
    ratio = geometry.loc[idx, 'z_spacing_mm'] / source_medians[source]
    z_outlier.loc[idx] = (ratio < 0.25) | (ratio > 4.0)
for row in geometry.loc[z_outlier].itertuples():
    evidence = {'z_spacing_mm': row.z_spacing_mm, 'source_median_mm': source_medians[row.source]}
    findings.append({
        'source': row.source, 'case_id': row.case_id, 'issue': 'Z_SPACING_OUTLIER',
        'evidence': json.dumps(evidence, ensure_ascii=False),
        'allowed_actions': 'KEEP_CONFIRMED|EXCLUDE',
    })

findings = pd.DataFrame(findings).sort_values(['source', 'case_id', 'issue']).reset_index(drop=True)
registry_path = RESULTS / '04_case_review_registry.csv'
editable = ['action', 'approved', 'reviewer', 'rationale']
if registry_path.is_file():
    previous = pd.read_csv(registry_path).fillna('')
    keep = ['source', 'case_id', 'issue', *editable]
    registry = findings.merge(previous[keep], on=['source', 'case_id', 'issue'], how='left')
else:
    registry = findings.copy()
    registry['action'] = ''
    registry['approved'] = False
    registry['reviewer'] = ''
    registry['rationale'] = ''
registry[editable] = registry[editable].fillna('')
registry.to_csv(registry_path, index=False, encoding='utf-8-sig')
display(registry)
print('검토 레지스트리:', registry_path)

,source,case_id,issue,evidence,allowed_actions,action,approved,reviewer,rationale
0,BHSD,ID_089596a9_ID_ea71b57820,AFFINE_MISMATCH,"{""affine_max_abs_diff"": 0.3871440922309828, ""i...",COPY_CT_AFFINE_TO_MASK|RESAMPLE_MASK_TO_CT|EXC...,,False,,
1,BHSD,ID_56faaf6f_ID_dd8f688dff,Z_SPACING_OUTLIER,"{""z_spacing_mm"": 0.5, ""source_median_mm"": 5.03...",KEEP_CONFIRMED|EXCLUDE,,False,,
2,BHSD,ID_727c946e_ID_e1fd84171a,AFFINE_MISMATCH,"{""affine_max_abs_diff"": 0.8620945796618293, ""i...",COPY_CT_AFFINE_TO_MASK|RESAMPLE_MASK_TO_CT|EXC...,,False,,
3,BHSD,ID_88dbe13f_ID_01c5078fbf,AFFINE_MISMATCH,"{""affine_max_abs_diff"": 0.8036462317956217, ""i...",COPY_CT_AFFINE_TO_MASK|RESAMPLE_MASK_TO_CT|EXC...,,False,,
4,BHSD,ID_a429fd9a_ID_2eabc43f3d,INTENSITY_SIGNED12,"{""min"": 0.0, ""p50"": 3096.0, ""p99"": 4078.0, ""ma...",APPLY_SIGNED12|KEEP_CONFIRMED|EXCLUDE,,False,,
5,BHSD,ID_cb4c887c_ID_ec2a4643f3,AFFINE_MISMATCH,"{""affine_max_abs_diff"": 0.5016892679503235, ""i...",COPY_CT_AFFINE_TO_MASK|RESAMPLE_MASK_TO_CT|EXC...,,False,,


검토 레지스트리: c:\workspace\hodgepodge\hy\데이터품질검증 및 통합\results\04_case_review_registry.csv


## 2. 통합 정책 초안

처음 실행하면 정책 초안을 만듭니다. 라이선스 확인, 출력 위치, 목표 spacing과 HU clipping 범위를 검토한 뒤 JSON을 수정하세요. 기존 파일이 있으면 덮어쓰지 않습니다.

In [4]:
policy_path = RESULTS / '04_integration_policy.json'
if not policy_path.is_file():
    policy = {
        'schema_version': 1,
        'task': 'binary_hemorrhage_segmentation',
        'binary_label_rule': 'scaled_mask > 0',
        'license_approved': False,
        'license_evidence': '',
        'target_orientation': 'RAS',
        'target_spacing_mm': [0.5, 0.5, 2.5],
        'hu_clip': [-1024.0, 3071.0],
        'split_ratios': {'train': 0.70, 'validation': 0.15, 'test': 0.15},
        'random_seed': 20260826,
        'processed_root': str(DATA_ROOT / 'integrated_binary_v1'),
        'max_mask_volume_relative_change': 0.35,
    }
    write_json(policy_path, policy)
else:
    policy = json.loads(policy_path.read_text(encoding='utf-8'))
print(json.dumps(policy, ensure_ascii=False, indent=2))

{
  "schema_version": 1,
  "task": "binary_hemorrhage_segmentation",
  "binary_label_rule": "scaled_mask > 0",
  "license_approved": false,
  "license_evidence": "",
  "target_orientation": "RAS",
  "target_spacing_mm": [
    0.5,
    0.5,
    5.0
  ],
  "hu_clip": [
    -1024.0,
    3071.0
  ],
  "split_ratios": {
    "train": 0.7,
    "validation": 0.15,
    "test": 0.15
  },
  "random_seed": 20260826,
  "processed_root": "C:\\Users\\duddl\\OneDrive\\바탕 화면\\CTdata\\integrated_binary_v1",
  "max_mask_volume_relative_change": 0.35
}


## 3. 승인 상태 검증

CSV의 `action`, `approved`, `reviewer`, `rationale`을 채운 뒤 이 셀을 다시 실행합니다. 원본 확인이 끝나지 않은 행은 뒤 노트북에서 `HOLD_REVIEW`로 유지됩니다.

In [7]:
registry = pd.read_csv(registry_path).fillna('')
allowed = {
    'INTENSITY_SIGNED12': {'APPLY_SIGNED12', 'KEEP_CONFIRMED', 'EXCLUDE'},
    'AFFINE_MISMATCH': {'COPY_CT_AFFINE_TO_MASK', 'RESAMPLE_MASK_TO_CT', 'EXCLUDE'},
    'Z_SPACING_OUTLIER': {'KEEP_CONFIRMED', 'EXCLUDE'},
}
registry['approved_bool'] = registry.approved.map(as_bool)
registry['valid_action'] = registry.apply(lambda row: row.action in allowed[row.issue], axis=1)
registry['complete_review'] = (
    registry.approved_bool & registry.valid_action
    & registry.reviewer.astype(str).str.strip().ne('')
    & registry.rationale.astype(str).str.strip().ne('')
)
unresolved = registry.loc[~registry.complete_review]
print('승인 완료:', int(registry.complete_review.sum()), '/', len(registry))
print('라이선스 승인:', as_bool(policy.get('license_approved', False)))
if len(unresolved):
    display(unresolved[['source', 'case_id', 'issue', 'action', 'approved', 'reviewer', 'rationale']])
else:
    print('모든 기술 예외의 조치가 승인되었습니다.')

승인 완료: 6 / 6
라이선스 승인: False
모든 기술 예외의 조치가 승인되었습니다.
